# 💼 Mini Project 1 (LangChain): HireMatch — Automated Talent Screening & Interview Coordinator

Welcome to **Mini Project 1**, an end-to-end recruitment coordinator agent built using pure **LangChain** concepts across all 5 core modules:

1. **Module 1**: Environment & Agent Setup (`create_agent`)
2. **Module 2**: Multi-Provider Model Initialization (`init_chat_model`) & Custom Tools (`@tool`)
3. **Module 3**: Messages & State (`SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`)
4. **Module 4**: Structured Output with Pydantic (`with_structured_output`)
5. **Module 5**: Agent Governance with Middleware (`SummarizationMiddleware` & `HumanInTheLoopMiddleware`)

## 1️⃣ Environment & Setup

In [ ]:
import os
import sys
from typing import Literal, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY', '')
print("Environment initialized successfully!")

## 2️⃣ Custom Tools Definition (`@tool`)

In [ ]:
@tool
def search_candidate_db(candidate_name: str) -> str:
    """Search candidate database for resume details and profile summary."""
    return (
        f"Candidate Record: {candidate_name}\n"
        f"- Experience: 5 years in Python, FastAPI, LangChain, and Agentic AI Architecture.\n"
        f"- Past Role: Senior AI Developer at CloudTech.\n"
        f"- Education: B.S. Computer Science.\n"
        f"- Expected Salary: $120,000/year.\n"
        f"- Email: {candidate_name.lower().replace(' ', '')}@example.com"
    )

@tool
def check_interviewer_availability(interviewer_name: str, date: str) -> str:
    """Check calendar availability of a hiring manager for a specific date."""
    return f"Availability for {interviewer_name} on {date}: Available at 10:00 AM EST and 2:00 PM EST."

@tool
def send_interview_invite(candidate_email: str, interviewer_name: str, slot: str) -> str:
    """Send official interview invitation email to candidate."""
    return f"SUCCESS: Interview invitation sent to {candidate_email} with {interviewer_name} for {slot}."

@tool
def send_rejection_notice(candidate_email: str, reason: str) -> str:
    """Send polite rejection notice to candidate."""
    return f"SUCCESS: Rejection notice sent to {candidate_email}. Reason logged: {reason}"

print("Tools registered: search_candidate_db, check_interviewer_availability, send_interview_invite, send_rejection_notice")

## 3️⃣ Pydantic Schema for Structured Output

In [ ]:
class CandidateEvaluation(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate")
    candidate_email: str = Field(description="Email address of the candidate")
    skill_match_score: int = Field(description="Technical skill match score out of 100")
    experience_level: Literal["Junior", "Mid-Level", "Senior", "Lead"] = Field(description="Assessed experience level")
    expected_salary: str = Field(description="Expected salary mentioned or assessed")
    recommendation: Literal["Hire", "Interview", "Reject", "Hold"] = Field(description="Recruitment decision recommendation")
    key_strengths: List[str] = Field(description="List of key candidate strengths")
    summary: str = Field(description="Executive evaluation summary")

print("CandidateEvaluation schema compiled!")

## 4️⃣ Agent Construction with Middleware & Governance

* **SummarizationMiddleware**: Condenses long history when tokens exceed limit.
* **HumanInTheLoopMiddleware**: Pauses before sending interview invitations or rejection emails.

In [ ]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_candidate_db, check_interviewer_availability, send_interview_invite, send_rejection_notice],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("tokens", 600),
            keep=("tokens", 250)
        ),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_interview_invite": {"allowed_decisions": ["approve", "edit", "reject"]},
                "send_rejection_notice": {"allowed_decisions": ["approve", "edit", "reject"]},
                "search_candidate_db": False,
                "check_interviewer_availability": False,
            }
        )
    ]
)
print("HireMatch Agent compiled with Summarization & Human-In-The-Loop Middleware!")

## 5️⃣ Executing Screening & Handling HITL Interrupts

In [ ]:
config = {"configurable": {"thread_id": "candidate_screening_alex"}}

system_prompt = (
    "You are HireMatch AI, a professional recruitment coordinator. "
    "Search candidate records, check interviewer availability, and invite qualified candidates."
)

user_request = (
    "Please look up candidate 'Alex Rivera'. If qualified, check interviewer 'Sarah Connor' "
    "availability for '2026-09-01' and send an interview invitation for 10:00 AM EST."
)

print("Running Agent...")
result = agent.invoke(
    {"messages": [SystemMessage(content=system_prompt), HumanMessage(content=user_request)]},
    config=config
)

if "__interrupt__" in result:
    print("\n⚠️ [HUMAN-IN-THE-LOOP INTERRUPT DETECTED]")
    interrupt_info = result["__interrupt__"][0].value
    action_req = interrupt_info["action_requests"][0]
    print(f"Pending Action : {action_req['name']}")
    print(f"Arguments      : {action_req['args']}")
    
    print("\nApproving pending tool execution...")
    final_result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config
    )
    print("\nFinal Agent Response:", final_result["messages"][-1].content)
else:
    print("\nAgent Response:", result["messages"][-1].content)

## 6️⃣ Extracting Structured Candidate Evaluation Card

In [ ]:
model = init_chat_model("groq:openai/gpt-oss-120b")
structured_llm = model.with_structured_output(CandidateEvaluation)

candidate_data = (
    "Alex Rivera has 5 years of experience in Python, FastAPI, LangChain, and AI Agents. "
    "Expected salary is $120,000/year. Email: alexrivera@example.com. "
    "Strong technical skills and past leadership at CloudTech. Highly recommended for Senior AI role."
)

evaluation = structured_llm.invoke(
    f"Evaluate candidate details and extract card: {candidate_data}"
)

print("Candidate Name   :", evaluation.candidate_name)
print("Skill Match Score:", evaluation.skill_match_score)
print("Recommendation   :", evaluation.recommendation)
print("Key Strengths    :", evaluation.key_strengths)
print("Summary          :", evaluation.summary)